In [1]:
import itertools
import csv
import random
from datetime import datetime
from transformers import pipeline

# 1) 定义 meta-templates
TEMPLATES = [
    "请你告知我 {ENTITY} 通常指的是什么/是什么意思/存在于哪里",
    "我想大致了解关于 {TOPIC} 的信息",
    "为什么会存在 {ENTITY} , 它的意义是什么",
    "我该如何处理/应对/操作/实现 {ATTRIBUTE} , 给我指导",

    " {ENTITY} 和 {TOPIC} 有哪些具体的差异",
    "在什么情况下会和 {ATTRIBUTE} 产生关系, 请举例说明"
]

# 2) 填充值表（你只要换这里就能跨领域）
ENTITIES = ["客户ID", "病人编号", "合同甲方", "员工姓名", "实验编号"]
TOPICS = ["交易记录", "诊断结果", "赔偿条款", "薪酬福利", "实验方法"]
ATTRIBUTES = ["义务", "治疗方案", "风险", "绩效", "数据来源"]

# 3) 简单 paraphraser（这里用 HuggingFace）
paraphraser = pipeline("text2text-generation", model="Vamsi/T5_Paraphrase_Paws", device=-1)

def paraphrase(text, num_return=3):
    """返回若干种改写"""
    outputs = paraphraser(text, num_return_sequences=num_return, num_beams=5, max_length=64)
    return [o["generated_text"] for o in outputs]

# 4) 生成 queries
candidates = []
for template in TEMPLATES:
    for e, t, a in itertools.product(ENTITIES, TOPICS, ATTRIBUTES):
        q = template.format(ENTITY=e, TOPIC=t, ATTRIBUTE=a)
        variants = paraphrase(q, num_return=2)
        all_qs = [q] + variants
        for v in all_qs:
            candidates.append({
                "template": template,
                "entity": e,
                "topic": t,
                "attribute": a,
                "query": v,
                "generated_time": datetime.now().isoformat()
            })

# 5) 保存 CSV
with open("universal_queries.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=candidates[0].keys())
    writer.writeheader()
    writer.writerows(candidates)

print(f"已生成 {len(candidates)} 条 universal queries，保存为 universal_queries.csv")


/workspace/zms/miniconda3/envs/rag-llm/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/workspace/zms/miniconda3/envs/rag-llm/lib/python3.10/site-packages/huggingface_hub/file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in /data/qiu_workspace/zms/models/huggingface/hub/models--Vamsi--T5_Paraphrase_Paws. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
  warnings.warn(message)


ValueError: Could not load model Vamsi/T5_Paraphrase_Paws with any of the following classes: (<class 'transformers.models.auto.modeling_auto.AutoModelForSeq2SeqLM'>, <class 'transformers.models.t5.modeling_t5.T5ForConditionalGeneration'>). See the original errors:

while loading with AutoModelForSeq2SeqLM, an error is thrown:
Traceback (most recent call last):
  File "/workspace/zms/miniconda3/envs/rag-llm/lib/python3.10/site-packages/transformers/modeling_utils.py", line 1023, in _get_resolved_checkpoint_files
    resolved_archive_file = cached_file(pretrained_model_name_or_path, filename, **cached_file_kwargs)
  File "/workspace/zms/miniconda3/envs/rag-llm/lib/python3.10/site-packages/transformers/utils/hub.py", line 312, in cached_file
    file = cached_files(path_or_repo_id=path_or_repo_id, filenames=[filename], **kwargs)
  File "/workspace/zms/miniconda3/envs/rag-llm/lib/python3.10/site-packages/transformers/utils/hub.py", line 557, in cached_files
    raise e
  File "/workspace/z